In [ ]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

In [ ]:
import os
from urllib.request import urlretrieve
import pandas as pd
import pgserver

from sqlalchemy import create_engine, text

pg = pgserver.get_server(
    '/tmp/hw7_pg',
    cleanup_mode='stop'
)

engine = create_engine(
    pg.get_uri(),
    future=True
)

with engine.connect() as conn:
    version = conn.execute(
        text("SELECT version()")
    ).scalar_one()

print(version)

In [48]:
URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = "/content/yellow_tripdata_2024-03.parquet"

urlretrieve(URL, parquet_path)

print("Downloaded!")

Downloaded!


In [49]:
columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(
    parquet_path,
    columns=columns
).head(500_000)

df.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0


In [50]:
df = df[
    (df['fare_amount'] > 0)
    & (df['fare_amount'] < 500)
]

df = df[
    (df['trip_distance'] > 0)
    & (df['trip_distance'] < 100)
]

df = df[
    df['PULocationID'].notna()
]

print(df.shape)

df.head()

(486964, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0


In [51]:
df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000
)

print("nyc_taxi_raw loaded!")

nyc_taxi_raw loaded!


In [52]:
pd.read_sql("""
    SELECT COUNT(*) AS n_rows
    FROM nyc_taxi_raw;
""", engine)

,n_rows
0,486964


In [57]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

        CREATE TABLE t7_hw_taxi_daily AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,
            DATE_TRUNC(
                'day',
                tpep_pickup_datetime::TIMESTAMP
            )::DATE AS ts_day,

            COUNT(*)::INTEGER AS trip_count,
            AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
            SUM(trip_distance)::NUMERIC(12, 2) AS total_distance

        FROM nyc_taxi_raw

        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100
          AND "PULocationID" IS NOT NULL

        GROUP BY
            "PULocationID"::INTEGER,
            DATE_TRUNC(
                'day',
                tpep_pickup_datetime::TIMESTAMP
            )::DATE;
    """))

print("t7_hw_taxi_daily created!")

t7_hw_taxi_daily created!


In [58]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE t7_hw_taxi_daily
        ADD PRIMARY KEY (
            pickup_zone_id,
            ts_day
        );
    """))

print("Primary key added!")

Primary key added!


In [59]:
daily_check = pd.read_sql("""
    SELECT
        pickup_zone_id,
        COUNT(*) AS n_days

    FROM t7_hw_taxi_daily

    GROUP BY pickup_zone_id

    ORDER BY pickup_zone_id

    LIMIT 10;
""", engine)

daily_check

,pickup_zone_id,n_days
0,1,5
1,3,6
2,4,6
3,6,4
4,7,6
5,8,1
6,9,5
7,10,6
8,11,4
9,12,5


In [60]:
pd.read_sql("""
    SELECT COUNT(*) AS n_rows
    FROM t7_hw_taxi_daily;
""", engine)

,n_rows
0,1217


### Daily taxi aggregation

Я створила таблицю `t7_hw_taxi_daily` на рівні pickup zone та дня.

Для кожного дня і зони розраховано:
- кількість поїздок;
- середню вартість поїздки;
- загальну дистанцію.

Ця таблиця буде використовуватися для наступних time-series features.

In [61]:
lag_features = pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,

        LAG(trip_count, 1) OVER w AS trips_lag_1d,

        LAG(trip_count, 7) OVER w AS trips_lag_7d,

        LAG(trip_count, 30) OVER w AS trips_lag_30d

    FROM t7_hw_taxi_daily

    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )

    ORDER BY
        pickup_zone_id,
        ts_day

    LIMIT 20;
""", engine)

lag_features

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,None,None
1,1,2024-03-02,2,1.0,None,None
2,1,2024-03-03,2,2.0,None,None
3,1,2024-03-04,1,2.0,None,None
4,1,2024-03-05,1,1.0,None,None
5,3,2024-03-01,3,NaN,None,None
6,3,2024-03-02,1,3.0,None,None
7,3,2024-03-03,4,1.0,None,None
8,3,2024-03-04,2,4.0,None,None
9,3,2024-03-05,1,2.0,None,None


### Lag features

Я створила lag-ознаки для кількості поїздок:

- `trips_lag_1d` — кількість поїздок 1 день тому;
- `trips_lag_7d` — кількість поїздок 7 днів тому;
- `trips_lag_30d` — кількість поїздок 30 днів тому.

`PARTITION BY pickup_zone_id` використовується для того, щоб історія кожної pickup zone рахувалася окремо.

In [62]:
rolling_7d = pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,

        AVG(avg_fare) OVER w7
            AS rolling_avg_fare_7d,

        COUNT(*) OVER w7
            AS days_in_window

    FROM t7_hw_taxi_daily

    WINDOW w7 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP

        RANGE BETWEEN
            INTERVAL '6 days' PRECEDING
            AND CURRENT ROW
    )

    ORDER BY
        pickup_zone_id,
        ts_day

    LIMIT 20;
""", engine)

rolling_7d

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-01,105.00,105.000000,1
1,1,2024-03-02,95.00,100.000000,2
2,1,2024-03-03,51.00,83.666667,3
3,1,2024-03-04,40.00,72.750000,4
4,1,2024-03-05,120.00,82.200000,5
5,3,2024-03-01,39.17,39.170000,1
6,3,2024-03-02,21.50,30.335000,2
7,3,2024-03-03,25.03,28.566667,3
8,3,2024-03-04,39.15,31.212500,4
9,3,2024-03-05,45.50,34.070000,5


### Rolling 7-day average fare

Я розрахувала середню вартість поїздки за останні 7 календарних днів для кожної pickup zone.

`PARTITION BY pickup_zone_id` гарантує, що кожна зона обчислюється окремо.

`RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` включає поточний день і попередні 6 днів.

In [63]:
rolling_30d = pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,

        STDDEV_SAMP(avg_fare) OVER w30
            AS rolling_std_fare_30d,

        COUNT(*) OVER w30
            AS days_in_window

    FROM t7_hw_taxi_daily

    WINDOW w30 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP

        RANGE BETWEEN
            INTERVAL '29 days' PRECEDING
            AND CURRENT ROW
    )

    ORDER BY
        pickup_zone_id,
        ts_day

    LIMIT 20;
""", engine)

rolling_30d

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-01,105.00,NaN,1
1,1,2024-03-02,95.00,7.071068,2
2,1,2024-03-03,51.00,28.728615,3
3,1,2024-03-04,40.00,32.045541,4
4,1,2024-03-05,120.00,34.881227,5
5,3,2024-03-01,39.17,NaN,1
6,3,2024-03-02,21.50,12.494577,2
7,3,2024-03-03,25.03,9.350841,3
8,3,2024-03-04,39.15,9.289451,4
9,3,2024-03-05,45.50,10.273605,5


### Rolling 30-day fare volatility

Я розрахувала стандартне відхилення середньої вартості поїздки за останні 30 календарних днів.

`STDDEV_SAMP` показує, наскільки сильно змінюється `avg_fare` у часі.

`PARTITION BY pickup_zone_id` використовується для того, щоб volatility рахувалася окремо для кожної pickup zone.

In [64]:
expanding_features = pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,

        SUM(trip_count) OVER we
            AS cumulative_trips,

        AVG(trip_count) OVER we
            AS expanding_avg_trips

    FROM t7_hw_taxi_daily

    WINDOW we AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day

        ROWS BETWEEN
            UNBOUNDED PRECEDING
            AND CURRENT ROW
    )

    ORDER BY
        pickup_zone_id,
        ts_day

    LIMIT 20;
""", engine)

expanding_features

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-01,1,1,1.000000
1,1,2024-03-02,2,3,1.500000
2,1,2024-03-03,2,5,1.666667
3,1,2024-03-04,1,6,1.500000
4,1,2024-03-05,1,7,1.400000
5,3,2024-03-01,3,3,3.000000
6,3,2024-03-02,1,4,2.000000
7,3,2024-03-03,4,8,2.666667
8,3,2024-03-04,2,10,2.500000
9,3,2024-03-05,1,11,2.200000


### Expanding cumulative features

Я розрахувала накопичувальну кількість поїздок і середню кількість поїздок від початку історії до поточного дня.

`UNBOUNDED PRECEDING` означає, що window починається з першого доступного рядка для кожної pickup zone.

`PARTITION BY pickup_zone_id` гарантує, що значення не змішуються між різними зонами.

In [65]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS

        WITH features_basic AS (
            SELECT
                pickup_zone_id,
                ts_day,
                trip_count,
                avg_fare,
                total_distance,

                LAG(trip_count, 1) OVER w_pid
                    AS trips_lag_1d,

                LAG(trip_count, 7) OVER w_pid
                    AS trips_lag_7d,

                LAG(trip_count, 30) OVER w_pid
                    AS trips_lag_30d,

                AVG(avg_fare) OVER w7
                    AS rolling_avg_fare_7d,

                STDDEV_SAMP(avg_fare) OVER w30
                    AS rolling_std_fare_30d,

                SUM(trip_count) OVER we
                    AS cumulative_trips,

                AVG(trip_count) OVER we
                    AS expanding_avg_trips

            FROM t7_hw_taxi_daily

            WINDOW

                w_pid AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day
                ),

                w7 AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day::TIMESTAMP

                    RANGE BETWEEN
                        INTERVAL '6 days' PRECEDING
                        AND CURRENT ROW
                ),

                w30 AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day::TIMESTAMP

                    RANGE BETWEEN
                        INTERVAL '29 days' PRECEDING
                        AND CURRENT ROW
                ),

                we AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day

                    ROWS BETWEEN
                        UNBOUNDED PRECEDING
                        AND CURRENT ROW
                )
        )

        SELECT *
        FROM features_basic

        WHERE trips_lag_30d IS NOT NULL;
    """))

print("Feature view created!")

Feature view created!


In [66]:
features_view = pd.read_sql("""
    SELECT *
    FROM v_t7_hw_taxi_features

    ORDER BY
        pickup_zone_id,
        ts_day

    LIMIT 20;
""", engine)

features_view

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips


### Combined time-series feature view

Я об'єднала всі time-series features в один view `v_t7_hw_taxi_features`.

У view входять lag-ознаки, rolling average, rolling volatility та expanding features.

У всіх window functions використовується `PARTITION BY pickup_zone_id`, щоб історія різних pickup zones не змішувалася.

Рядки без 30-денної історії відфільтровані через `trips_lag_30d IS NOT NULL`.

In [67]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_events;

        CREATE TABLE t7_hw_events (
            event_id INTEGER
                GENERATED BY DEFAULT AS IDENTITY
                PRIMARY KEY,

            user_id INTEGER NOT NULL,

            event_ts TIMESTAMPTZ
                NOT NULL DEFAULT NOW(),

            event_type TEXT NOT NULL,

            payload JSONB NOT NULL
        );
    """))

print("t7_hw_events created!")

t7_hw_events created!


In [68]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO t7_hw_events (
            user_id,
            event_ts,
            event_type,
            payload
        )

        SELECT
            (n % 1000) + 1 AS user_id,

            TIMESTAMPTZ '2024-04-01 00:00:00+00'
                - ((n % 60) || ' days')::INTERVAL AS event_ts,

            CASE (n % 5)
                WHEN 0 THEN 'page_view'
                WHEN 1 THEN 'click'
                WHEN 2 THEN 'add_to_cart'
                WHEN 3 THEN 'checkout'
                ELSE 'recommendation_shown'
            END AS event_type,

            CASE (n % 5)

                WHEN 0 THEN jsonb_build_object(
                    'url', '/product/' || ((n % 200) + 1),
                    'referrer',
                        (ARRAY['google','direct','email','fb'])
                        [(n % 4) + 1],
                    'device',
                        (ARRAY['mobile','desktop','tablet'])
                        [(n % 3) + 1]
                )

                WHEN 1 THEN jsonb_build_object(
                    'element_id', 'btn_' || ((n % 50) + 1),
                    'page_url', '/category/' || ((n % 20) + 1),
                    'device',
                        (ARRAY['mobile','desktop'])
                        [(n % 2) + 1]
                )

                WHEN 2 THEN jsonb_build_object(
                    'product_id', (n % 200) + 1,
                    'price', ((n % 100) + 10)::NUMERIC(10, 2),
                    'quantity', (n % 5) + 1,
                    'tags',
                        jsonb_build_array(
                            (ARRAY['promo','new','clearance'])
                            [(n % 3) + 1]
                        )
                )

                WHEN 3 THEN jsonb_build_object(
                    'order_id', 'o_' || (n + 100000),
                    'total', ((n % 500) + 50)::NUMERIC(10, 2),
                    'items_count', (n % 10) + 1,
                    'payment_method',
                        (ARRAY['card','paypal','crypto'])
                        [(n % 3) + 1]
                )

                ELSE jsonb_build_object(
                    'model_name', 'rec_v' || ((n % 5) + 1),
                    'model_version', '1.' || (n % 10),

                    'recommendations',
                        jsonb_build_array(
                            (n % 200) + 1,
                            ((n + 1) % 200) + 1,
                            ((n + 2) % 200) + 1
                        ),

                    'experiment',
                        'exp_' || ((n % 3) + 1)
                )

            END AS payload

        FROM generate_series(1, 50000) AS n;
    """))

print("50,000 events inserted!")

50,000 events inserted!


In [69]:
with engine.begin() as conn:
    conn.execute(text("""
        ANALYZE t7_hw_events;
    """))

events_check = pd.read_sql("""
    SELECT
        event_type,
        COUNT(*) AS n_events

    FROM t7_hw_events

    GROUP BY event_type

    ORDER BY event_type;
""", engine)

events_check

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### JSONB event-log

Я створила таблицю `t7_hw_events` для зберігання подій користувачів.

Таблиця містить 5 типів подій:
`page_view`, `click`, `add_to_cart`, `checkout` та `recommendation_shown`.

Поле `payload` має тип `JSONB` і містить різні поля залежно від типу події.

Такий підхід зручний для event-data, де структура подій може відрізнятися.

"payment_method": "card"

In [70]:
jsonb_query_1 = pd.read_sql("""
    SELECT
        event_id,
        user_id,
        event_ts,
        payload

    FROM t7_hw_events

    WHERE event_type = 'checkout'

      AND payload @> jsonb_build_object(
          'payment_method',
          'card'
      )

    LIMIT 10;
""", engine)

jsonb_query_1

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### JSONB Query 1 — Containment

У цьому запиті я використала оператор `@>` для пошуку JSONB-об'єктів, які містять потрібну пару ключ-значення.

Запит знаходить checkout-події, де `payment_method` дорівнює `card`.

Оператор `@>` зручний для перевірки, чи JSONB payload містить певні дані.

In [71]:
jsonb_query_2a = pd.read_sql("""
    SELECT
        event_id,
        event_type,
        payload

    FROM t7_hw_events

    WHERE payload ? 'tags'

    LIMIT 10;
""", engine)

jsonb_query_2a

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [72]:
jsonb_query_2b = pd.read_sql("""
    SELECT
        event_id,
        event_type,
        payload

    FROM t7_hw_events

    WHERE payload ?| ARRAY[
        'experiment',
        'model_version'
    ]

    LIMIT 10;
""", engine)

jsonb_query_2b

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### JSONB Query 2 — Key existence

У першому запиті я використала оператор `?`, щоб перевірити, чи JSONB payload містить ключ `tags`.

У другому запиті використовується оператор `?|`, який перевіряє, чи присутній хоча б один ключ зі списку.

Такі оператори зручні для пошуку подій із різною структурою JSONB.

In [73]:
jsonb_query_3a = pd.read_sql("""
    SELECT
        event_id,

        jsonb_path_query(
            payload,
            '$.recommendations[*]'
        ) AS recommended_pid

    FROM t7_hw_events

    WHERE event_type = 'recommendation_shown'

    LIMIT 15;
""", engine)

jsonb_query_3a

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [ ]:
jsonb_query_3b = pd.read_sql("""
    SELECT
        event_id,

        payload ->> 'order_id'
            AS order_id,

        (payload ->> 'total')::NUMERIC(10, 2)
            AS total

    FROM t7_hw_events

    WHERE event_type = 'checkout'

      AND jsonb_path_exists(
          payload,
          '$.total ? (@ > 300)'
      )

    LIMIT 10;
""", engine)

jsonb_query_3b

### JSONB Query 3 — JSONPath

У першому запиті я використала `jsonb_path_query`, щоб отримати окремі значення з масиву `recommendations`.

У другому запиті використовується `jsonb_path_exists`, щоб знайти checkout-події, де значення `total` більше 300.

JSONPath зручний для роботи зі складнішими структурами всередині JSONB.

In [74]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
        DROP INDEX IF EXISTS idx_t7_events_device_btree;

        ANALYZE t7_hw_events;
    """))

print("Indexes removed and table analyzed!")

Indexes removed and table analyzed!


In [75]:
explain_before_gin = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT COUNT(*)

    FROM t7_hw_events

    WHERE payload @> jsonb_build_object(
        'payment_method',
        'card'
    );
""", engine)

explain_before_gin

,QUERY PLAN
0,Aggregate (cost=1758.25..1758.26 rows=1 width...
1,Buffers: shared hit=1000
2,-> Seq Scan on t7_hw_events (cost=0.00..17...
3,Filter: (payload @> jsonb_build_object...
4,Rows Removed by Filter: 46666
5,Buffers: shared hit=1000
6,Planning:
7,Buffers: shared hit=5
8,Planning Time: 0.181 ms
9,Execution Time: 48.506 ms


### EXPLAIN до GIN index

Я виконала containment-запит через оператор `@>` до створення GIN index.

На цьому етапі PostgreSQL може використовувати `Seq Scan`, тобто переглядати таблицю послідовно.

Цей output буде використаний пізніше для порівняння з планом після створення GIN index.

In [76]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_payload_gin_pathops
            ON t7_hw_events
            USING GIN (payload jsonb_path_ops);

        ANALYZE t7_hw_events;
    """))

print("GIN index created!")

GIN index created!


In [77]:
gin_size = pd.read_sql("""
    SELECT
        pg_size_pretty(
            pg_relation_size(
                'idx_t7_events_payload_gin_pathops'
            )
        ) AS gin_index_size;
""", engine)

gin_size

,gin_index_size
0,1144 kB


Bitmap Index Scan
Bitmap Heap Scan

In [78]:
explain_gin_referrer = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT COUNT(*)

    FROM t7_hw_events

    WHERE payload @> jsonb_build_object(
        'referrer',
        'google'
    );
""", engine)

explain_gin_referrer

,QUERY PLAN
0,Aggregate (cost=1065.69..1065.70 rows=1 width...
1,Buffers: shared hit=1004
2,-> Bitmap Heap Scan on t7_hw_events (cost=...
3,Recheck Cond: (payload @> jsonb_build_...
4,Heap Blocks: exact=1000
5,Buffers: shared hit=1004
6,-> Bitmap Index Scan on idx_t7_events...
7,Index Cond: (payload @> jsonb_bu...
8,Buffers: shared hit=4
9,Planning:


### EXPLAIN після GIN index

Після створення GIN index я повторила containment-запит через оператор `@>`.

Тепер PostgreSQL може використовувати GIN index замість повного послідовного читання таблиці.

Я порівнюю цей план із попереднім EXPLAIN за фактичними операторами, cost та actual time.

In [79]:
explain_device_before = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT COUNT(*)

    FROM t7_hw_events

    WHERE payload ->> 'device' = 'mobile';
""", engine)

explain_device_before

,QUERY PLAN
0,Aggregate (cost=1750.62..1750.63 rows=1 width...
1,Buffers: shared hit=1000
2,-> Seq Scan on t7_hw_events (cost=0.00..17...
3,Filter: ((payload ->> 'device'::text) ...
4,Rows Removed by Filter: 41667
5,Buffers: shared hit=1000
6,Planning Time: 0.073 ms
7,Execution Time: 22.892 ms


In [80]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_device_btree

        ON t7_hw_events (
            (payload ->> 'device')
        );

        ANALYZE t7_hw_events;
    """))

print("Device expression index created!")

Device expression index created!


In [81]:
explain_device_after = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT COUNT(*)

    FROM t7_hw_events

    WHERE payload ->> 'device' = 'mobile';
""", engine)

explain_device_after

,QUERY PLAN
0,Aggregate (cost=1242.42..1242.43 rows=1 width...
1,Buffers: shared hit=2 read=9
2,-> Bitmap Heap Scan on t7_hw_events (cost=...
3,Recheck Cond: ((payload ->> 'device'::...
4,Heap Blocks: exact=1000
5,Buffers: shared hit=2 read=9
6,-> Bitmap Index Scan on idx_t7_events...
7,Index Cond: ((payload ->> 'devic...
8,Buffers: shared read=9
9,Planning:


### Expression index для device

Запит `payload ->> 'device' = 'mobile'` працює з конкретним текстовим значенням усередині JSONB.

GIN index на всьому `payload` не є найкращим індексом для такого виразу.

Тому я створила expression B-tree index на `(payload ->> 'device')` і повторила EXPLAIN ANALYZE.

Тепер можна порівняти actual time, cost і оператори плану до та після індексу.

In [82]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_checkouts_norm;

        CREATE TABLE t7_hw_checkouts_norm (
            event_id INTEGER PRIMARY KEY,
            user_id INTEGER NOT NULL,
            event_ts TIMESTAMPTZ NOT NULL,

            order_id TEXT NOT NULL,

            total NUMERIC(10, 2) NOT NULL,

            items_count INTEGER NOT NULL
                CHECK (items_count > 0),

            payment_method TEXT NOT NULL
                CHECK (
                    payment_method IN (
                        'card',
                        'paypal',
                        'crypto'
                    )
                )
        );
    """))

print("Normalized checkout table created!")

Normalized checkout table created!


In [83]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO t7_hw_checkouts_norm (
            event_id,
            user_id,
            event_ts,
            order_id,
            total,
            items_count,
            payment_method
        )

        SELECT
            event_id,
            user_id,
            event_ts,

            payload ->> 'order_id'
                AS order_id,

            (payload ->> 'total')::NUMERIC(10, 2)
                AS total,

            (payload ->> 'items_count')::INTEGER
                AS items_count,

            payload ->> 'payment_method'
                AS payment_method

        FROM t7_hw_events

        WHERE event_type = 'checkout';
    """))

print("Checkout data inserted!")

Checkout data inserted!


In [84]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_checkouts_norm_pm
            ON t7_hw_checkouts_norm (
                payment_method
            );

        ANALYZE t7_hw_checkouts_norm;
    """))

print("Checkout index created!")

Checkout index created!


In [85]:
pd.read_sql("""
    SELECT COUNT(*) AS n_checkouts
    FROM t7_hw_checkouts_norm;
""", engine)

,n_checkouts
0,10000


### Normalized checkout table

Я створила окрему нормалізовану таблицю для checkout-подій.

Замість зберігання всіх даних у JSONB, основні поля checkout тепер зберігаються як окремі typed columns.

Це спрощує SQL-запити, constraints та індексацію для стабільної структури даних.

In [86]:
explain_jsonb_checkout = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT
        payload ->> 'payment_method' AS payment_method,

        AVG(
            (payload ->> 'total')::NUMERIC
        ) AS avg_total,

        COUNT(*) AS n

    FROM t7_hw_events

    WHERE event_type = 'checkout'

    GROUP BY
        payload ->> 'payment_method'

    ORDER BY
        avg_total DESC

    LIMIT 3;
""", engine)

explain_jsonb_checkout

,QUERY PLAN
0,Limit (cost=1954.79..1954.80 rows=3 width=72)...
1,Buffers: shared hit=1000
2,-> Sort (cost=1954.79..1968.65 rows=5545 w...
3,Sort Key: (avg(((payload ->> 'total'::...
4,Sort Method: quicksort Memory: 25kB
5,Buffers: shared hit=1000
6,-> HashAggregate (cost=1799.95..1883...
7,Group Key: (payload ->> 'payment...
8,Batches: 1 Memory Usage: 217kB
9,Buffers: shared hit=1000


In [88]:
print(
    "\n".join(
        explain_jsonb_checkout.iloc[:, 0].astype(str)
    )
)

Limit  (cost=1954.79..1954.80 rows=3 width=72) (actual time=25.035..25.038 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1954.79..1968.65 rows=5545 width=72) (actual time=25.033..25.035 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.95..1883.12 rows=5545 width=72) (actual time=24.982..25.021 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.99 rows=9997 width=132) (actual time=0.015..15.577 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=16
Planning Time: 0.193 ms
Execution Time: 25.141 ms


In [89]:
explain_norm_checkout = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT
        payment_method,

        AVG(total) AS avg_total,

        COUNT(*) AS n

    FROM t7_hw_checkouts_norm

    GROUP BY
        payment_method

    ORDER BY
        avg_total DESC

    LIMIT 3;
""", engine)

explain_norm_checkout

,QUERY PLAN
0,Limit (cost=269.06..269.07 rows=3 width=46) (...
1,Buffers: shared hit=94
2,-> Sort (cost=269.06..269.07 rows=3 width=...
3,Sort Key: (avg(total)) DESC
4,Sort Method: quicksort Memory: 25kB
5,Buffers: shared hit=94
6,-> HashAggregate (cost=269.00..269.0...
7,Group Key: payment_method
8,Batches: 1 Memory Usage: 24kB
9,Buffers: shared hit=94


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=explain_norm_checkout)

### JSONB vs normalized

Я виконала однаковий аналітичний запит двома способами.

У JSONB-версії PostgreSQL спочатку витягує значення `payment_method` і `total` з поля `payload`.

У normalized-версії ці значення вже зберігаються в окремих typed columns.

Порівняння `EXPLAIN ANALYZE` покаже, який варіант був швидшим у моєму середовищі та які оператори використовував PostgreSQL.

In [90]:
# 1. BEFORE GIN
explain_before_gin = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object(
        'payment_method', 'card'
    );
""", engine)


# 2. AFTER GIN
explain_after_gin = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object(
        'payment_method', 'card'
    );
""", engine)


# 3. DEVICE BEFORE / current plan
explain_device_before = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
""", engine)


# 4. DEVICE AFTER B-TREE
explain_device_after = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
""", engine)


# 5. JSONB CHECKOUT
explain_jsonb_checkout = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT
        payload ->> 'payment_method' AS payment_method,
        AVG((payload ->> 'total')::NUMERIC) AS avg_total,
        COUNT(*) AS n

    FROM t7_hw_events

    WHERE event_type = 'checkout'

    GROUP BY payload ->> 'payment_method'

    ORDER BY avg_total DESC

    LIMIT 3;
""", engine)


# 6. NORMALIZED CHECKOUT
explain_norm_checkout = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT
        payment_method,
        AVG(total) AS avg_total,
        COUNT(*) AS n

    FROM t7_hw_checkouts_norm

    GROUP BY payment_method

    ORDER BY avg_total DESC

    LIMIT 3;
""", engine)

print("All EXPLAIN results created!")

All EXPLAIN results created!


In [91]:
print("=== 1. BEFORE GIN ===")
print("\n".join(explain_before_gin.iloc[:, 0].astype(str)))

print("\n=== 2. AFTER GIN ===")
print("\n".join(explain_after_gin.iloc[:, 0].astype(str)))

print("\n=== 3. DEVICE BEFORE B-TREE ===")
print("\n".join(explain_device_before.iloc[:, 0].astype(str)))

print("\n=== 4. DEVICE AFTER B-TREE ===")
print("\n".join(explain_device_after.iloc[:, 0].astype(str)))

print("\n=== 5. JSONB CHECKOUT ===")
print("\n".join(explain_jsonb_checkout.iloc[:, 0].astype(str)))

print("\n=== 6. NORMALIZED CHECKOUT ===")
print("\n".join(explain_norm_checkout.iloc[:, 0].astype(str)))

=== 1. BEFORE GIN ===
Aggregate  (cost=1129.38..1129.39 rows=1 width=8) (actual time=6.827..6.829 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=42.90..1117.03 rows=4942 width=0) (actual time=0.710..6.601 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..41.66 rows=4942 width=0) (actual time=0.520..0.520 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.163 ms
Execution Time: 6.864 ms

=== 2. AFTER GIN ===
Aggregate  (cost=1129.38..1129.39 rows=1 width=8) (actual time=5.354..5.356 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=42.90..1117.03 rows=4942 width=0) (actual time=0.

In [92]:
# BEFORE B-TREE
with engine.begin() as conn:
    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_device_btree;
        ANALYZE t7_hw_events;
    """))

device_before_real = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
""", engine)

print("=== DEVICE BEFORE B-TREE ===")
print("\n".join(device_before_real.iloc[:, 0].astype(str)))


# CREATE B-TREE AGAIN
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_device_btree
            ON t7_hw_events ((payload ->> 'device'));

        ANALYZE t7_hw_events;
    """))


# AFTER B-TREE
device_after_real = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
""", engine)

print("\n=== DEVICE AFTER B-TREE ===")
print("\n".join(device_after_real.iloc[:, 0].astype(str)))

=== DEVICE BEFORE B-TREE ===
Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=19.620..19.623 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.013..18.705 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.106 ms
Execution Time: 19.650 ms

=== DEVICE AFTER B-TREE ===
Aggregate  (cost=1242.07..1242.08 rows=1 width=8) (actual time=3.599..3.602 rows=1 loops=1)
  Buffers: shared hit=2 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.68..1221.30 rows=8308 width=0) (actual time=0.753..2.778 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=2 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.60 rows=8308 width=0) (

## Reflection

Під час виконання домашнього завдання я порівняла роботу JSONB-запитів до і після створення індексів.

Для containment-запиту через `@>` до GIN index PostgreSQL використовував `Seq Scan`. Час виконання був приблизно **48.728 ms**. Після створення GIN index план змінився на `Bitmap Index Scan` та `Bitmap Heap Scan`, а час виконання зменшився приблизно до **5.360 ms**. У моєму випадку GIN index значно прискорив пошук усередині JSONB.

GIN добре працює для операторів типу `@>`, тому що він індексує структуру і значення JSONB. Але для запиту `payload ->> 'device' = 'mobile'` краще підходить expression B-tree index. До такого індексу PostgreSQL використовував `Seq Scan` з часом **10.010 ms**, а після створення B-tree index — `Bitmap Index Scan` та `Bitmap Heap Scan` з часом приблизно **2.685 ms**.

Індекси покращують швидкість читання, але мають trade-off. Вони займають додаткове місце і можуть робити `INSERT` та `UPDATE` дорожчими, тому що PostgreSQL також повинен оновлювати індекс.

При порівнянні JSONB та normalized таблиці JSONB-запит виконався за **12.555 ms**, а normalized-версія — за **3.776 ms**. У цьому випадку normalized таблиця була швидшою. Я б використовувала JSONB для подій із різною або часто змінюваною структурою. Для стабільних даних із важливими типами, constraints та частими аналітичними запитами я б обрала normalized таблицю.

Якщо потрібно додати новий event-type `refund`, у JSONB-підході достатньо додати новий `event_type` і нову структуру `payload`. У normalized-підході, скоріше за все, потрібно створити нову таблицю або додати нові колонки та виконати schema migration.

GIN:
before = 48.728 ms, Seq Scan
after  = 5.360 ms, Bitmap Index Scan + Bitmap Heap Scan

JSONB checkout:
12.555 ms

Normalized checkout:
3.776 ms